# Build database of spatial footprints for input CSDA Program Evaluation Imagery

| Author | Affiliation | Date |
| ---------------- | ---------------- | ---------------- |
| Paul Montesano, PhD | Innovation Lab ; NASA Goddard Space Flight Center | Nov. 2025 |

In [1]:
#pip install matplotlib_scalebar

In [2]:
import geopandas as gpd
import pandas as pd
from urllib.parse import quote
import numpy as np
import os, sys

sys.path.append('/home/pmontesa/code/geoscitools')

sys.path.append('/home/pmontesa/code/csda_summaries/lib')

In [3]:
import glob
import importlib
import re
import os
from pathlib import Path
from collections import Counter
import pandas as pd
import geopandas as gpd

import csdalib_refactored

# If sensor_profiles.py is separate
import sensor_profiles
importlib.reload(sensor_profiles)

/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3

<module 'sensor_profiles' from '/panfs/ccds02/home/pmontesa/code/csda_summaries/notebooks/sensor_profiles.py'>

### Check which sensor profiles are available for making footprints

In [4]:
from csdalib_refactored import SENSOR_PROFILES, detect_vendor

print('Loaded profiles:')
for k, v in SENSOR_PROFILES.items():
    print(f"  {k:15s} → {v['affiliation']} / {v.get('constellation', '<dynamic>')}")

Loaded profiles:
  tanager         → Planet / Tanager
  legion          → Maxar / Legion
  maxar_imd       → Maxar / <dynamic>
  maxar_legacy    → Maxar / <dynamic>
  pixxel_fpt      → Pixxel / Firefly
  pixxel          → Pixxel / Firefly
  airbus_dimap    → Airbus / <dynamic>
  satellogic      → Satellogic / Aleph-1


## Build list of files to footprint

In [5]:
# Get today's date as a pandas Timestamp object
today_timestamp = pd.Timestamp.today()

# Convert the Timestamp object to a string in a specific format (e.g., YYYY-MM-DD)
today_date_string = today_timestamp.strftime('%Y-%m-%d')

## Search

In [6]:
# ---- 1. Configure search ----
ROOT_DIR = '/explore/nobackup/projects/CSDA_eval'
VENDOR_SUBDIRS = [
    'Satellogic',
    'Planet',
    'Pixxel', 
    'Maxar',
    'Airbus'
]

SEARCH_PATTERNS = ['*.TIF', '*.tif', '*.JP2', '*.jp2', '*.json', '*.geojson']

In [7]:
# ---- 2. Find candidate files per vendor subdir ----
all_files = []
for sub in VENDOR_SUBDIRS:
    for pat in SEARCH_PATTERNS:
        hits = glob.glob(f'{ROOT_DIR}/{sub}/**/{pat}', recursive=True)
        all_files.extend(hits)

print(f'Found {len(all_files)} candidate files across {len(VENDOR_SUBDIRS)} vendor subdirs')

Found 4748 candidate files across 5 vendor subdirs


In [8]:
# ---- 3. Skip noise files (ie preview and other files that shouldnt be footprinted) ----
SKIP_PATTERNS = [
    r'-request\.json$',
    r'_manifest\.json$',
    r'README',
    r'^CCS_', r'^CLD_', r'^DCL_', r'^HAZ_', r'^MSK_', r'^GFP_',
    r'_TCI\.tif$',   # ← Pixxel True Color Image preview (not primary data)
    r'_TCI\.jpeg$',
    r'_PCL\.tif$',   # ← Pixxel Product Classification Layer (also auxiliary)
]

def is_skip(path):
    fn = Path(path).name.lower()
    return any(re.search(p, fn, re.IGNORECASE) for p in SKIP_PATTERNS)

n_before = len(all_files)
all_files = [f for f in all_files if not is_skip(f)]
print(f'Filtered out {n_before - len(all_files)} noise files')
print(f'Remaining files to process: {len(all_files)}')

Filtered out 479 noise files
Remaining files to process: 4269


In [9]:
# ---- 4. Pre-filter: keep only files where dispatcher detects a vendor ----
detected = []
unmatched = []
for f in all_files:
    profile = csdalib_refactored.detect_vendor(f)
    if profile:
        detected.append((f, profile))
    else:
        unmatched.append(f)

print(f'Recognized: {len(detected)} files')
print(f'Unmatched:  {len(unmatched)} files')

print('\nFiles per detected profile:')
for p, n in Counter(p for _, p in detected).most_common():
    print(f'  {p:20s}: {n}')

Recognized: 4240 files
Unmatched:  29 files

Files per detected profile:
  airbus_dimap        : 2382
  legion              : 1222
  satellogic          : 360
  pixxel              : 129
  pixxel_fpt          : 109
  tanager             : 38


In [10]:
unmatched

['/explore/nobackup/projects/CSDA_eval/Maxar/Legion/25APR16165718-M1BS-200007822894_01_P001.TIF',
 '/explore/nobackup/projects/CSDA_eval/Maxar/Legion/25APR16165718-P1BS-200007822894_01_P001.TIF',
 '/explore/nobackup/projects/CSDA_eval/Maxar/Legion/25APR10165131-M1BS-200007713473_01_P001.TIF',
 '/explore/nobackup/projects/CSDA_eval/Maxar/Legion/25APR10165131-P1BS-200007713473_01_P001.TIF',
 '/explore/nobackup/projects/CSDA_eval/Maxar/Legion/25APR09203435-M1BS-200007703077_01_P001.TIF',
 '/explore/nobackup/projects/CSDA_eval/Maxar/Legion/25APR09203435-P1BS-200007703077_01_P001.TIF',
 '/explore/nobackup/projects/CSDA_eval/Maxar/Legion/25APR09170559-M1BS-200007700894_01_P001.TIF',
 '/explore/nobackup/projects/CSDA_eval/Maxar/Legion/25APR09170559-P1BS-200007700894_01_P001.TIF',
 '/explore/nobackup/projects/CSDA_eval/Maxar/Legion/delivery/B120001102664A00/200009790953_01/200009790953_01_P001_PAN/25SEP13130134-P1BS-200009790953_01_P001.TIF',
 '/explore/nobackup/projects/CSDA_eval/Maxar/Legion

In [14]:
#csdalib_refactored.detect_vendor('/explore/nobackup/projects/CSDA_eval/Maxar/Legion/25APR16165718-M1BS-200007822894_01_P001.TIF', profiles=SENSOR_PROFILES)()

# Footprint

In [15]:
# ============================================================================
# Footprinting Configuration
# ============================================================================

FOOTPRINT_BASE_FN = '/explore/nobackup/projects/CSDA_eval/footprints/footprints_CSDA_eval_'
footprint_gdf_FN = f'{FOOTPRINT_BASE_FN}SCENES_{today_date_string}.gpkg'
footprint_ACQS_gdf_FN = f'{FOOTPRINT_BASE_FN}ACQUISITIONS_{today_date_string}.gpkg'

In [16]:
importlib.reload(csdalib_refactored)

# ---- 5. Process all detected files ----
files_to_process = [f for f, _ in detected]
footprint_gdf = csdalib_refactored.process_files(files_to_process)

print(f'\nBuilt GeoDataFrame with {len(footprint_gdf)} rows')

# ---- 6. Summary ----
print('\nFootprints per vendor profile:')
print(footprint_gdf['vendor_profile'].value_counts())
print('\nFootprints per affiliation:')
print(footprint_gdf['affiliation'].value_counts())
print('\nFootprints per sensor:')
print(footprint_gdf['sensor'].value_counts().head(20))

Error processing FF03_20260319_00501045_0000007394_L2A.tif: '/explore/nobackup/projects/CSDA_eval/Pixxel/V2/FF03_20260319_00501045_0000007394_L2A.tif' not recognized as a supported file format.
Error processing IMG_PNEO4_HD_202510051040574_PMS_ORT_PWOI_000501695_1_1_F_1_RGBN_R1C1.TIF: /explore/nobackup/projects/CSDA_eval/Airbus/HD15/000501695_1_1_HD_A/IMG_01_PNEO4_PMS/IMG_PNEO4_HD_202510051040574_PMS_ORT_PWOI_000501695_1_1_F_1_RGBN_R1C1.TIF: TIFFReadDirectory:Failed to read directory at offset 1677721608

Built GeoDataFrame with 4129 rows

Footprints per vendor profile:
airbus_dimap    2381
legion          1222
satellogic       360
pixxel           128
tanager           38
Name: vendor_profile, dtype: int64

Footprints per affiliation:
Airbus        2381
Maxar         1222
Satellogic     360
Pixxel         128
Planet          38
Name: affiliation, dtype: int64

Footprints per sensor:
PNEO3        699
PNEO4        633
PHR1A        393
LG01         374
SPOT6        336
PHR1B        320
L

### Footprint checks

In [17]:
print('Acquisition ID sources:')
print(footprint_gdf['acquisition_id_source'].value_counts())

Acquisition ID sources:
canonical    4129
Name: acquisition_id_source, dtype: int64


In [18]:
print('Acquisitions with fallback IDs (no canonical metadata ID):')
fallback_acqs = footprint_gdf[footprint_gdf['acquisition_id'].str.startswith('FALLBACK_')]
print(f'  Count: {len(fallback_acqs)}')
print(f'  By vendor:\n{fallback_acqs["vendor_profile"].value_counts()}')
print(f'  Sample paths:\n{fallback_acqs["file_path"].head(5).tolist()}')

Acquisitions with fallback IDs (no canonical metadata ID):
  Count: 0
  By vendor:
Series([], Name: vendor_profile, dtype: int64)
  Sample paths:
[]


In [19]:
import importlib, sensor_profiles, csdalib_refactored
importlib.reload(sensor_profiles); importlib.reload(csdalib_refactored)

from csdalib_refactored import SENSOR_PROFILES
print(SENSOR_PROFILES['tanager'].get('acquisition_id_field'))

image_id


In [20]:
print('=== Per-vendor acquisition_id health ===\n')

for prof in footprint_gdf['vendor_profile'].unique():
    sub = footprint_gdf[footprint_gdf['vendor_profile'] == prof]
    n_total    = len(sub)
    n_fallback = (sub['acquisition_id_source'] == 'fallback').sum()
    n_no_meta  = (~sub['has_metadata']).sum()
    n_acqs     = sub['acquisition_id'].nunique()
    print(f'{prof:18s}  files={n_total:5d}  acqs={n_acqs:5d}  '
          f'fallback={n_fallback:3d}  missing_xml={n_no_meta:3d}')

print(f'\nTotal files:        {len(footprint_gdf)}')
print(f'Total acquisitions: {footprint_gdf["acquisition_id"].nunique()}')
print(f'Any fallback?       {(footprint_gdf["acquisition_id_source"] == "fallback").sum()}')

=== Per-vendor acquisition_id health ===

satellogic          files=  360  acqs=   60  fallback=  0  missing_xml=  0
tanager             files=   38  acqs=   19  fallback=  0  missing_xml=  0
pixxel              files=  128  acqs=  109  fallback=  0  missing_xml= 18
legion              files= 1222  acqs=   78  fallback=  0  missing_xml=  0
airbus_dimap        files= 2381  acqs=  266  fallback=  0  missing_xml=  0

Total files:        4129
Total acquisitions: 532
Any fallback?       0


## Read sites GeoJson
best to work downstream of this notebook that creates a GeoJSON AOI of our sites

In [21]:
csda_sites_fn = f'/explore/nobackup/projects/CSDA_eval/sites/csda_sites_aoi.geojson'

In [22]:
sites_gdf = gpd.read_file(csda_sites_fn)

In [23]:
### Buffer MORE for sites for display
BUF_KM_ADD_FOR_DISPLAY = 25
BUF_KM_TOTAL_FOR_DISPLAY = BUF_KM_ADD_FOR_DISPLAY #+ BUF_KM

In [24]:
sites_gdf_buf = sites_gdf.to_crs(3857).buffer(BUF_KM_ADD_FOR_DISPLAY * 1000) # add arbitrary buffer to evaluation sites for map display
sites_gdf_buf_display = gpd.GeoDataFrame(sites_gdf.drop(columns=['geometry']), geometry=sites_gdf_buf, crs=sites_gdf_buf.crs).to_crs(4326)
sites_gdf_buf_display = sites_gdf_buf_display[~(sites_gdf_buf_display.geometry.is_empty | sites_gdf_buf_display.geometry.isna())]

## Link footprints of acquisitions to Evaluation Sites

#### using `dissolve` and `join`

In [25]:
importlib.reload(csdalib_refactored)

<module 'csdalib_refactored' from '/panfs/ccds02/home/pmontesa/code/csda_summaries/notebooks/csdalib_refactored.py'>

In [26]:
import csdalib

In [27]:
# Display summary
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

#if not footprint_gdf.empty:
print(f"\nSuccessful footprints: {len(footprint_gdf)}")
# print("\nBreakdown by type:")
# print(footprint_gdf.groupby('source_type').size())

# Save combined footprints
footprint_gdf.to_file(footprint_gdf_FN, driver='GPKG')

print(f"\n✓ Saved combined SCENE footprints to: {footprint_gdf_FN}")

# Dissolve scenes to acquisitions (combining all scenes and band combos)
footprint_gdf_acquisitions = footprint_gdf.dissolve(
    by='acquisition_id',
    aggfunc={
        #'base_image_name': lambda x: list(set(x)),
        #'path': lambda x: list(set(x)),
        'sensor': 'first',
        'affiliation': 'first',
        'constellation': 'first',
        'image_type': lambda x: list(set([b for b in x if b])),
        # 'catid': 'first',
        'acquisition_datetime': 'first',
        'year': 'first',
        'month': 'first',
        'day': 'first',
        'scene_id': lambda x: list(set(x)),  # Unique scene IDs
        'band_variant':  lambda x: list(set([b for b in x if b])),  
        #'band_combo': lambda x: list(set([b for b in x if b])),  # Unique band combos (excluding None)
        #'file': 'count'  # Total count of files: where are image scenes x image band combinations
    }
).reset_index()

# Put 'Site Name' on ACQUISITIONS footprint: Link acquisitions to sites with buffer
# Needed : calcs 'num_sites'
footprint_gdf_acquisitions_sites, acq_site_mapping = csdalib.link_acquisitions_to_sites(
    
    footprint_gdf_acquisitions.to_crs(3857),  # Your acquisition-level footprints
    sites_gdf, # this is already buffered
    buffer_distance=1000,  # 1 km buffer (adjust as needed)
    #site_name_col_primary = 'site_primary',
    site_name_col='Site Name'
)

footprint_gdf_acquisitions_sites = csdalib.prepare_gdf_for_export(footprint_gdf_acquisitions_sites)

footprint_gdf_acquisitions_sites.to_file(footprint_ACQS_gdf_FN, driver='GPKG')
footprint_gdf_acquisitions_sites.to_csv(footprint_ACQS_gdf_FN.replace('gpkg','csv'))

print(f"\n✓ Saved combined ACQUISITIONS footprints to: {footprint_ACQS_gdf_FN}")


FINAL SUMMARY

Successful footprints: 4129

✓ Saved combined SCENE footprints to: /explore/nobackup/projects/CSDA_eval/footprints/footprints_CSDA_eval_SCENES_2026-07-01.gpkg
Consider reprojecting to a projected CRS for accurate buffering.
Reprojecting sites to match footprints CRS...

Summary:
  Total acquisitions: 532
  Acquisitions intersecting sites: 503
  Acquisitions NOT intersecting sites: 29
  Single-site acquisitions: 471
  Multi-site acquisitions (2+ sites): 32
  Acquisitions covering 3+ sites: 7
Converting column 'image_type' from list to string
Converting column 'scene_id' from list to string
Converting column 'band_variant' from list to string
Converting column 'sites' from list to string

✓ Saved combined ACQUISITIONS footprints to: /explore/nobackup/projects/CSDA_eval/footprints/footprints_CSDA_eval_ACQUISITIONS_2026-07-01.gpkg


In [28]:
footprint_gdf_acquisitions_sites.shape

(532, 17)

In [29]:
footprint_gdf.head()

,file_path,metadata_path,vendor_profile,affiliation,constellation,sensor,image_type,image_id,sensor_id_raw,gsd,...,view_angle,satellite_azimuth,item_type,quality_category,catid,sensor_image_id,_needs_raster_footprint,mission,mission_index,num_bands
0,/explore/nobackup/projects/CSDA_eval/Satellogi...,/explore/nobackup/projects/CSDA_eval/Satellogi...,satellogic,Satellogic,Aleph-1,SN10,MS,20210202_090141_SN10_L1D_MS_None,newsat10,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,/explore/nobackup/projects/CSDA_eval/Satellogi...,/explore/nobackup/projects/CSDA_eval/Satellogi...,satellogic,Satellogic,Aleph-1,SN10,MS,20210202_090141_SN10_L1D_MS_None,newsat10,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,/explore/nobackup/projects/CSDA_eval/Satellogi...,/explore/nobackup/projects/CSDA_eval/Satellogi...,satellogic,Satellogic,Aleph-1,SN10,MS,20210202_090141_SN10_L1D_MS_None,newsat10,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,/explore/nobackup/projects/CSDA_eval/Satellogi...,/explore/nobackup/projects/CSDA_eval/Satellogi...,satellogic,Satellogic,Aleph-1,SN10,MS,20210205_091047_SN10_L1D_MS_27152,newsat10,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,/explore/nobackup/projects/CSDA_eval/Satellogi...,/explore/nobackup/projects/CSDA_eval/Satellogi...,satellogic,Satellogic,Aleph-1,SN10,MS,20210205_091047_SN10_L1D_MS_27152,newsat10,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
#footprint_gdf_acquisitions_sites[footprint_gdf_acquisitions_sites.sensor == 'Tanager-1'].explore()

In [31]:
footprint_gdf_acquisitions_sites.tail(2)

,acquisition_id,geometry,sensor,affiliation,constellation,image_type,acquisition_datetime,year,month,day,scene_id,band_variant,Site_Primary,Site_Secondary,Site_Tertiary,num_sites,sites
530,SEN_SPOT6_20260123_045336100_000,"MULTIPOLYGON (((-106.89780 35.29603, -106.3363...",SPOT6,Airbus,SPOT,"P, MS",2026-01-23T00:00:00+00:00,2026,1,23,"R1C1, R1C2, R2C1, R2C2",,Albuquerque,Shadnagar,None,2,"Albuquerque, Shadnagar"
531,SEN_SPOT6_20260123_045437100_000,"MULTIPOLYGON (((-106.89780 35.29603, -106.3363...",SPOT6,Airbus,SPOT,"P, MS",2026-01-23T00:00:00+00:00,2026,1,23,"R1C1, R1C2, R2C1, R2C2",,Albuquerque,Shadnagar,None,2,"Albuquerque, Shadnagar"


In [32]:
#footprint_gdf_acquisitions_sites.tail(2).explore()

In [33]:
m = csdalib.make_CSDA_footprints_map(footprint_gdf_acquisitions_sites#[footprint_gdf_acquisitions_sites.image_type != 'PAN']#.drop('acquisition_datetime', axis=1).to_crs(4326)
                                 #, height='25%'
                                 , ACQS=True, site_name_col='sites'
                                 , MAP = sites_gdf.explore(style_kwds=dict(style_function=csdalib.dashed_style), name='Sites (actual)', 
                                                           m=sites_gdf_buf_display.explore(style_kwds=dict(style_function=csdalib.dashed_style), name='Sites (display)')),
                                 TOOLTIP_FIELDS_LIST = ['affiliation','constellation','sensor','acquisition_id','year','month','day']
                                )

QML file created: footprints_CSDA_eval_ACQUISITIONS_combined_label_color_dict.qml


### Make HTML for webmap of footprints

In [34]:
out_html = '/home/pmontesa/code/csda_summaries/notebooks/footprints_csda_map.html'

In [35]:
# Save to html

In [43]:
import os

# 1. Save the map as usual
m.save(out_html)

# 2. Build the header HTML (title + image + description)
header_html = """
    <!-- Outer card container -->
    <div style='padding: 28px 36px; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Arial, sans-serif; max-width: 1100px; margin: 24px auto; color: #222; background: #91bfdb; border: 1px solid #e0e6ee; border-radius: 10px; box-shadow: 0 2px 6px rgba(31, 77, 140, 0.06);'>
      
      <!-- Eyebrow tag -->
      <div style='display: inline-block; background: #fee090; color: #1f4d8c; padding: 4px 10px; border-radius: 4px; font-size: 0.78em; font-weight: 600; letter-spacing: 0.04em; text-transform: uppercase; margin-bottom: 10px;'>
        CSDA Evaluation Image Footprints
      </div>
      
      <!-- Title -->
      <h1 style='color: #1f4d8c; font-size: 1.85em; line-height: 1.25; margin: 0 0 6px 0; font-weight: 600;'>
        Interactive map of footprints of images available for CSDA SME Evaluations
      </h1>
      
      <!-- Subtitle/lede -->
      <p style='font-size: 1em; line-height: 1.55; margin: 6px 0 18px 0; color: #4a5568;'>
        These are imagery delivered in response to CSDA requests at sites from the <a href='https://docs.google.com/spreadsheets/d/13MrpqFtAOqQY9WdW9lHNsqjCbG-e3VQkEDbHOGIKa6k/edit?gid=538266839#gid=538266839' target='_blank' style='color: #1f4d8c; text-decoration: none; border-bottom: 2px solid #cfdcef;'>CSDA Evaluation Sites Database</a>.
      </p>
      
      <!-- Key features list -->
      <div style='background: #f6f9fd; border-left: 3px solid #1f4d8c; padding: 12px 16px; margin: 16px 0; border-radius: 0 4px 4px 0;'>
        <p style='margin: 0 0 8px 0; font-size: 0.95em; font-weight: 600; color: #1f4d8c;'>Each image footprint:</p>
        <ul style='margin: 0; padding-left: 20px; font-size: 0.95em; line-height: 1.55; color: #333;'>
          <li>is located at a CSDA Evaluation Site named using a unique identifier (<code style='background:#fff; padding:1px 5px; border-radius:3px; font-size:0.9em; border: 1px solid #e0e6ee;'>Site Name</code>)</li>
          <li>has geometric bounds representing the acauisition extent</li>
          <li>includes basic attributes associated with the image acquisition</li>
        </ul>
      </div>
      
      <!-- Display note -->
      <p style='font-size: 0.95em; line-height: 1.55; margin: 14px 0; color: #4a5568;'>
        Image footprint geometry should <b>completely cover</b> an site AOI. Mouse over footprints for acquisition details.
      </p>
      
      <!-- Footer row: status + contact -->
      <div style='display: flex; justify-content: space-between; align-items: center; margin-top: 18px; padding-top: 12px; border-top: 1px solid #e0e6ee; font-size: 0.85em; color: #718096;'>
        <span style='font-style: italic;'>Work in progress — some images were not properly footprinted and are missing from this map.</span>
        <span><b>Contact:</b> <a href='mailto:paul.m.montesano@nasa.gov' style='color: #1f4d8c; text-decoration: none;'>paul.m.montesano@nasa.gov</a></span>
      </div>
      
    </div>
"""


# 3. CSS to make the folium map fill the page width and a tall portion of the viewport
fullwidth_css = """
<style>
  body { margin: 0; padding: 0; }
  /* Folium map container — override fixed width/height */
  .folium-map {
    width: 100% !important;
    height: 45vh !important;
    max-width: 100% !important;
  }
  /* If folium wraps the map in a Figure div, also stretch it */
  div[id^="map_"] {
    width: 100% !important;
    height: 45vh !important;
  }
</style>
"""

# 4. Inject header + CSS right after <body>
with open(out_html, 'r') as f:
    html = f.read()

html = html.replace('<body>', f'<body>\n{fullwidth_css}\n{header_html}', 1)

with open(out_html, 'w') as f:
    f.write(html)

print(f"Saved {out_html} — push to GitHub to view at:")
print(f"  https://pahbs.github.io/maap_tools/maps/{out_html}")


Saved /home/pmontesa/code/csda_summaries/notebooks/footprints_csda_map.html — push to GitHub to view at:
  https://pahbs.github.io/maap_tools/maps//home/pmontesa/code/csda_summaries/notebooks/footprints_csda_map.html


## Plot maps of sites and their acqs (static)

In [29]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import contextily as ctx

import matplotlib.patches as mpatches
from matplotlib.dates import DateFormatter, MonthLocator, WeekdayLocator

### Explore XML to learn about fields in order to update `sensor_profiles.py`

In [30]:
import xml.etree.ElementTree as ET

# Pick a sample XML
xml_path = '/explore/nobackup/projects/CSDA_eval/Pixxel/V2/FF02_20260404_00501045_0000008716_L2A.xml'

# These separate images have same Image_ID ... bad
xml_path_list = [
    '/explore/nobackup/projects/CSDA_eval/Pixxel/V2/FF01_20251104_00501045_0000002446_L2A.xml',
    '/explore/nobackup/projects/CSDA_eval/Pixxel/V2/FF02_20251025_00501045_0000002446_L2A.xml'
]
for xml_path in xml_path_list:
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    # Strip namespaces
    for elem in root.iter():
        if '}' in elem.tag:
            elem.tag = elem.tag.split('}', 1)[1]
    
    # Print all leaf elements (the actual data tags)
    def walk(elem, path=''):
        tag = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        cur = f'{path}/{tag}' if path else tag
        if len(list(elem)) == 0:   # leaf node
            text = (elem.text or '').strip()
            if text:
                print(f'  {cur}: {text[:60]}')
        for child in elem:
            walk(child, cur)
    
    walk(root)
    print('\n')

  MetaData/Satellite_Details/Satellite: Pixxel-FF01
  MetaData/Satellite_Details/Sensor_Name: VNIR
  MetaData/Image_Acquisition_Details/Acquisition_Datetime: 2025-11-04T16:54:16Z
  MetaData/Image_Acquisition_Details/Image_ID: 0000002446
  MetaData/Image_Acquisition_Details/Altitude: 595.94
  MetaData/Image_Acquisition_Details/Off_Nadir_Angle: 1.72
  MetaData/Image_Acquisition_Details/Scene_Center_Lat: 30.41812897
  MetaData/Image_Acquisition_Details/Scene_Center_Lon: -86.83856201
  MetaData/Image_Acquisition_Details/Sun_Azimuth_Angle: 167.9
  MetaData/Image_Acquisition_Details/Sun_Elevation_Angle: 43.16
  MetaData/Image_Acquisition_Details/Earth_Sun_Distance: 0.9918
  MetaData/Image_Acquisition_Details/Number_Of_Bands: 45
  MetaData/Image_Acquisition_Details/Bands: B005, B009, B010, B011, B012, B013, B014, B015, B016, B017, 
  MetaData/Image_Acquisition_Details/Central_Wavelength: 485.9, 501.2, 505.2, 509.2, 512.6, 516.4, 520.5, 524.6, 528.
  MetaData/Image_Acquisition_Details/Bandwidt

In [31]:
footprint_gdf[footprint_gdf.acquisition_id == '0000002446'].metadata_path.to_list()

[]

### Strange case of SPOT images with same ID (Albuquerque & Shadnagar)

Now the picture is clear. This is a real data anomaly: two physically distinct SPOT6 acquisitions have the exact same filenames (IMG_SPOT6_MS_202601230453359_SEN_7674265101_R1C1.TIF):  

First 8 files: Shadnagar, India (17.0°N, 78.2°E) in /Airbus/SPOT/shadnagar/...  
Second 8 files: New Mexico (35.1°N, -106.6°W) in /Airbus/PROD_SPOT6_001/...  
Same timestamp (202601230453359), same CATID (7674265101), same product code. Either:  

Airbus genuinely delivered two different acquisitions with the same identifier (vendor metadata bug)  
The vendor reused a CATID for two physically different deliveries (also a vendor issue)  
You have a folder copied twice on disk (your data layout issue)  
Either way, the canonical metadata field (DATASET_NAME / JOB_ID, which gives SEN_SPOT6_20260123_045336100_000) is not actually unique across deliveries in your data. So acquisition_id alone can't disambiguate them.  

In [32]:
# Filter for records that contain a comma in the sites column
multiple_sites_gdf = footprint_gdf_acquisitions_sites[footprint_gdf_acquisitions_sites['sites'].str.contains(',', na=False)]
multiple_sites_gdf

,acquisition_id,geometry,sensor,affiliation,constellation,image_type,acquisition_datetime,year,month,day,scene_id,band_variant,Site_Primary,Site_Secondary,Site_Tertiary,num_sites,sites,combined_label
8,200009817627_01_P001,"POLYGON ((4.97110 43.35251, 4.97277 43.40389, ...",LG01,Maxar,Legion,"P, MS",2025-08-25T10:40:12+00:00,2025,8,25,"R13C2, R09C1, R07C2, R09C3, R03C2, R06C1, R07C...",,Etang de Berre,Salon-de-Provence,None,2,"Etang de Berre, Salon-de-Provence",Maxar - Legion
164,DS_PHR1A_202508221045161_FR1_PX_E004N43_1114_0...,"POLYGON ((4.97863 43.46969, 4.93356 43.47046, ...",PHR1A,Airbus,Pleiades,"P, MS",2025-08-22T00:00:00+00:00,2025,8,22,"R1C1, R1C2, R2C2, R2C1",,La Crau,La Crau TIR,None,2,"La Crau, La Crau TIR",Airbus - Pleiades
176,DS_PHR1A_202509091823434_FR1_PX_W113N33_1013_0...,"POLYGON ((-112.14638 33.41441, -112.18558 33.4...",PHR1A,Airbus,Pleiades,"P, MS",2025-09-09T00:00:00+00:00,2025,9,9,"R1C1, R1C2, R2C2, R2C1",,AZ Maricopa Pinal,Phoenix,None,2,"AZ Maricopa Pinal, Phoenix",Airbus - Pleiades
182,DS_PHR1A_202510051823484_FR1_PX_W113N33_1012_0...,"POLYGON ((-112.35748 33.40597, -112.35776 33.4...",PHR1A,Airbus,Pleiades,"P, MS",2025-10-05T00:00:00+00:00,2025,10,5,"R1C1, R1C2, R2C2, R2C1",,AZ Maricopa Pinal,Phoenix,None,2,"AZ Maricopa Pinal, Phoenix",Airbus - Pleiades
185,DS_PHR1A_202510271038015_FR1_PX_E004N43_1114_0...,"POLYGON ((4.98964 43.46368, 4.92304 43.46482, ...",PHR1A,Airbus,Pleiades,"P, MS",2025-10-27T00:00:00+00:00,2025,10,27,"R1C1, R1C2, R2C2, R2C1",,La Crau,La Crau TIR,None,2,"La Crau, La Crau TIR",Airbus - Pleiades
186,DS_PHR1A_202510311824030_FR1_PX_W113N33_1012_0...,"POLYGON ((-112.13882 33.40766, -112.19256 33.4...",PHR1A,Airbus,Pleiades,"P, MS",2025-10-31T00:00:00+00:00,2025,10,31,"R1C1, R1C2, R2C2, R2C1",,AZ Maricopa Pinal,Phoenix,None,2,"AZ Maricopa Pinal, Phoenix",Airbus - Pleiades
189,DS_PHR1A_202511221037366_FR1_PX_E004N43_1114_0...,"POLYGON ((4.99033 43.46378, 4.92263 43.46494, ...",PHR1A,Airbus,Pleiades,"P, MS",2025-11-22T00:00:00+00:00,2025,11,22,"R1C1, R1C2, R2C2, R2C1",,La Crau,La Crau TIR,None,2,"La Crau, La Crau TIR",Airbus - Pleiades
221,DS_PHR1B_202510071041331_FR1_PX_E004N43_1114_0...,"POLYGON ((4.98865 43.46163, 4.92318 43.46275, ...",PHR1B,Airbus,Pleiades,"P, MS",2025-10-07T00:00:00+00:00,2025,10,7,"R1C1, R1C2, R2C2, R2C1",,La Crau,La Crau TIR,None,2,"La Crau, La Crau TIR",Airbus - Pleiades
229,DS_PHR1B_202512051037246_FR1_PX_E004N43_1114_0...,"POLYGON ((4.99034 43.46468, 4.92179 43.46585, ...",PHR1B,Airbus,Pleiades,"P, MS",2025-12-05T00:00:00+00:00,2025,12,5,"R1C1, R1C2, R2C2, R2C1",,La Crau,La Crau TIR,None,2,"La Crau, La Crau TIR",Airbus - Pleiades
230,DS_PHR1B_202512171045011_FR1_PX_E004N43_1114_0...,"POLYGON ((4.98941 43.46129, 4.92225 43.46244, ...",PHR1B,Airbus,Pleiades,"P, MS",2025-12-17T00:00:00+00:00,2025,12,17,"R1C1, R1C2, R2C2, R2C1",,La Crau,La Crau TIR,None,2,"La Crau, La Crau TIR",Airbus - Pleiades


In [33]:
# Define sites to plot
sites_to_plot = sorted([site for site in footprint_gdf_acquisitions_sites.Site_Primary.unique() if site != 'Not CSDA Eval Site'])
sites_to_plot

['AZ Maricopa Pinal',
 'Albuquerque',
 'Baotou',
 'Belo Horizonte',
 'Boston',
 'Cape Town',
 'Casablanca',
 'Caspian Sea',
 'Catania',
 'Crater Lake',
 'Cuprite',
 'Etang de Berre',
 'Gobabeb',
 'Hohhot',
 'King Fahd Causeway',
 'Konya',
 'La Crau',
 'London',
 'Melbourne',
 'Navarre Causeway',
 'PICS Algeria-3',
 'PICS Libya-4',
 'Piedmont',
 'Railroad Valley',
 'Rio Gallegos',
 'Salon-de-Provence',
 'Sapporo',
 'Shadnagar',
 'Valencia',
 'WLEF']

In [34]:
 #footprint_gdf_acquisitions_sites[footprint_gdf_acquisitions_sites.Site_Primary.isin(sites_to_plot)].explore()

In [35]:
import pandas as pd

dup_check = (
    footprint_gdf.to_crs(3857)
    .groupby('acquisition_id')
    .agg(
        n_sensors=('sensor', 'nunique'),
        n_dates  =('day',    'nunique'),
        n_lats   =('geometry', lambda g: g.centroid.y.round(0).nunique()),
        sensors  =('sensor', lambda x: ','.join(sorted(set(x)))),
        n_files  =('file_path', 'count'),
        profile  =('vendor_profile', 'first'),
    )
    .reset_index()
)

ambiguous = dup_check[
    (dup_check['n_sensors'] > 1) |
    (dup_check['n_dates']   > 1) |
    (dup_check['n_lats']    > 1)
]
print(f'Total acquisition_ids: {len(dup_check)}')
print(f'Ambiguous (multi-sensor/date/location): {len(ambiguous)}')
print('\nBy vendor profile:')
print(ambiguous['profile'].value_counts())
print('\nSample of worst cases:')
print(ambiguous.sort_values('n_files', ascending=False).head(10).to_string())

Total acquisition_ids: 532
Ambiguous (multi-sensor/date/location): 307

By vendor profile:
airbus_dimap    229
legion           78
Name: profile, dtype: int64

Sample of worst cases:
          acquisition_id  n_sensors  n_dates  n_lats sensors  n_files profile
8   200009817627_01_P001          1        1      43    LG01       82  legion
4   200009817616_01_P001          1        1      32    LG03       62  legion
29  200010867962_01_P001          1        1      32    LG01       60  legion
5   200009817622_01_P001          1        1      27    LG03       52  legion
2   200009817608_01_P001          1        1      28    LG04       50  legion
52  200010964783_01_P001          1        1      24    LG02       44  legion
32  200010867968_01_P001          1        1      17    LG05       32  legion
54  200010964787_01_P001          1        1      29    LG01       29  legion
56  200010965767_01_P001          1        1      29    LG05       29  legion
57  200010965772_01_P001          1  

In [36]:
# Pick the worst Legion case: 'B130001100CDD100' (66 files, LG03 + Maxar)
sample_acq = footprint_gdf[footprint_gdf['acquisition_id'] == 'B130001100CDD100']

# What sensors come up?
print('Files per sensor in this acquisition:')
print(sample_acq['sensor'].value_counts())

# What metadata path do they have?
print('\nFiles with sensor=Maxar (falling back from LG03):')
maxar_fallback = sample_acq[sample_acq['sensor'] == 'Maxar']
print(maxar_fallback[['file_path', 'metadata_path', 'has_metadata']].head(5).to_string())

# Compare against the LG03-resolved files
print('\nFiles with sensor=LG03 (resolved correctly):')
lg03_ok = sample_acq[sample_acq['sensor'] == 'LG03']
print(lg03_ok[['file_path', 'metadata_path', 'has_metadata']].head(5).to_string())

Files per sensor in this acquisition:
Series([], Name: sensor, dtype: int64)

Files with sensor=Maxar (falling back from LG03):
Empty DataFrame
Columns: [file_path, metadata_path, has_metadata]
Index: []

Files with sensor=LG03 (resolved correctly):
Empty DataFrame
Columns: [file_path, metadata_path, has_metadata]
Index: []


In [37]:
# # One M2AS file (the "Maxar" sensor)
# m2as = footprint_gdf[
#     (footprint_gdf['acquisition_id'] == 'B130001100CDD100') &
#     (footprint_gdf['sensor'] == 'Maxar')
# ].iloc[0]

# print('M2AS file full record:')
# for k, v in m2as.items():
#     print(f'  {k:30s} = {v!r}')

In [38]:
print('M2AS files in the dataset:')
m2as_all = footprint_gdf[
    footprint_gdf['file_path'].str.contains('M2AS_R')
]
print(f'Total M2AS files: {len(m2as_all)}')
print('\nTheir catids:')
print(m2as_all['catid'].value_counts())
print('\nTheir vendor_profiles:')
print(m2as_all['vendor_profile'].value_counts())
print('\nTheir paths:')
print(m2as_all['file_path'].head(3).tolist())

M2AS files in the dataset:
Total M2AS files: 60

Their catids:
200008787224_01_P001    6
200008787244_01_P001    6
200012660305_01_P001    4
200012660268_01_P001    4
200012660223_01_P001    4
200012660272_01_P001    4
200012660299_01_P001    4
200012660302_01_P001    4
200012660192_01_P001    4
200012660138_01_P001    4
200012660116_01_P001    4
200012660092_01_P001    4
200012660164_01_P001    4
200012979330_01_P001    4
Name: catid, dtype: int64

Their vendor_profiles:
legion    60
Name: vendor_profile, dtype: int64

Their paths:
['/explore/nobackup/projects/CSDA_eval/Maxar/Legion/delivery/B120001101D41700/200008787224_01/200008787224_01_P001_MUL/25JUN08220602-M2AS_R3C2-200008787224_01_P001.TIF', '/explore/nobackup/projects/CSDA_eval/Maxar/Legion/delivery/B120001101D41700/200008787224_01/200008787224_01_P001_MUL/25JUN08220602-M2AS_R3C1-200008787224_01_P001.TIF', '/explore/nobackup/projects/CSDA_eval/Maxar/Legion/delivery/B120001101D41700/200008787224_01/200008787224_01_P001_MUL/25JU

In [39]:
import xml.etree.ElementTree as ET

xml_paths = [
    '/explore/nobackup/projects/CSDA_eval/Maxar/Legion/delivery/B130001100CDD100/200009817616_01/200009817616_01_P001_MUL/25MAY16162848-M3DS-200009817616_01_P001.XML',
    '/explore/nobackup/projects/CSDA_eval/Maxar/Legion/OR2A/Valencia/25MAY16-M2AS_200012660092_01_P001_MUL/25MAY16162850-M2AS-200012660092_01_P001.XML',
]

for xml_path in xml_paths:
    print('=' * 80)
    print(xml_path)
    print('=' * 80)
    tree = ET.parse(xml_path)
    root = tree.getroot()
    for elem in root.iter():
        if '}' in elem.tag:
            elem.tag = elem.tag.split('}', 1)[1]
    # Print any tag related to CATID or order
    for elem in root.iter():
        tag = elem.tag.upper()
        if any(k in tag for k in ['CATID', 'ORDER', 'CATALOG', 'SCENE_ID', 'IMAGE_ID', 'DATASET']):
            if elem.text and elem.text.strip():
                print(f'  <{elem.tag}>: {elem.text.strip()}')

/explore/nobackup/projects/CSDA_eval/Maxar/Legion/delivery/B130001100CDD100/200009817616_01/200009817616_01_P001_MUL/25MAY16162848-M3DS-200009817616_01_P001.XML
  <PRODUCTORDERID>: 200009817616_01_P001
  <PRODUCTCATALOGID>: None
  <CATID>: B130001100CDD100
/explore/nobackup/projects/CSDA_eval/Maxar/Legion/OR2A/Valencia/25MAY16-M2AS_200012660092_01_P001_MUL/25MAY16162850-M2AS-200012660092_01_P001.XML
  <PRODUCTORDERID>: 200012660092_01_P001
  <PRODUCTCATALOGID>: None
  <CATID>: B130001100CDD100


In [40]:
# lg03 = footprint_gdf[
#     (footprint_gdf['acquisition_id'] == 'B130001100CDD100') &
#     (footprint_gdf['sensor'] == 'LG03')
# ].iloc[0]

# print('\nLG03 file full record:')
# for k, v in lg03.items():
#     print(f'  {k:30s} = {v!r}')

In [41]:
import sys
for m in list(sys.modules):
    if 'csdalib' in m or 'sensor_profiles' in m:
        del sys.modules[m]

import csdalib_refactored
from csdalib_refactored import process_files, SENSOR_PROFILES

# Confirm the changes
print('Legion patterns:', SENSOR_PROFILES['legion']['detect']['filename_patterns'])
print('\nLegion catid spec:', SENSOR_PROFILES['legion']['fields']['catid'])

Legion patterns: ['-[PMS]3DS_R\\d+C\\d+', '-[PM]2[AS]S_R\\d+C\\d+', '\\d{2}[A-Z]{3}\\d{8}-[PMS][23][AD]S']

Legion catid spec: {'tag': 'PRODUCTORDERID', 'fallback_tags': ['PRODUCTCATALOGID', 'CATID']}


In [42]:
# Check the previously-problematic acquisition
prev = footprint_gdf[footprint_gdf['acquisition_id'].str.contains('200009817616')]
print(f'Files for 200009817616: {len(prev)}')
print(f'  Sensors: {prev["sensor"].value_counts().to_dict()}')

# Check that the M2AS files now route correctly
m2as = footprint_gdf[footprint_gdf['file_path'].str.contains('M2AS_R')]
print(f'\nM2AS files: {len(m2as)}')
print(f'  Now in profile: {m2as["vendor_profile"].value_counts().to_dict()}')
print(f'  Sensor: {m2as["sensor"].value_counts().to_dict()}')

# Re-run ambiguity check
from collections import Counter
dup_check = (
    footprint_gdf
    .groupby('acquisition_id')
    .agg(n_sensors=('sensor', 'nunique'),
         n_lats=('geometry', lambda g: g.centroid.y.round(0).nunique()))
)
print(f'\nAmbiguous acquisitions (multi-sensor or multi-lat): '
      f'{(dup_check["n_sensors"] > 1).sum() + (dup_check["n_lats"] > 1).sum()}')

Files for 200009817616: 62
  Sensors: {'LG03': 62}

M2AS files: 60
  Now in profile: {'legion': 60}
  Sensor: {'LG02': 20, 'LG01': 20, 'LG03': 12, 'LG04': 8}


/explore/nobackup/people/pmontesa/.nccstmp/ipykernel_2261459/1207497933.py:18: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  n_lats=('geometry', lambda g: g.centroid.y.round(0).nunique()))
/explore/nobackup/people/pmontesa/.nccstmp/ipykernel_2261459/1207497933.py:18: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  n_lats=('geometry', lambda g: g.centroid.y.round(0).nunique()))
/explore/nobackup/people/pmontesa/.nccstmp/ipykernel_2261459/1207497933.py:18: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  n_lats=('geometry', lambda g: g.centroid.y.round(0).nunique()))
/explore/n


Ambiguous acquisitions (multi-sensor or multi-lat): 76


/explore/nobackup/people/pmontesa/.nccstmp/ipykernel_2261459/1207497933.py:18: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  n_lats=('geometry', lambda g: g.centroid.y.round(0).nunique()))
/explore/nobackup/people/pmontesa/.nccstmp/ipykernel_2261459/1207497933.py:18: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  n_lats=('geometry', lambda g: g.centroid.y.round(0).nunique()))
/explore/nobackup/people/pmontesa/.nccstmp/ipykernel_2261459/1207497933.py:18: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  n_lats=('geometry', lambda g: g.centroid.y.round(0).nunique()))
/explore/n

In [43]:
import pandas as pd

dup_check = (
    footprint_gdf
    .groupby('acquisition_id')
    .agg(
        n_sensors=('sensor',   'nunique'),
        n_dates  =('day',      'nunique'),
        n_lats   =('geometry', lambda g: g.centroid.y.round(0).nunique()),
        sensors  =('sensor',   lambda x: ','.join(sorted(set(x)))),
        n_files  =('file_path','count'),
        profile  =('vendor_profile','first'),
    )
    .reset_index()
)

ambiguous = dup_check[
    (dup_check['n_sensors'] > 1) |
    (dup_check['n_dates']   > 1) |
    (dup_check['n_lats']    > 1)
]

print(f'Ambiguous: {len(ambiguous)}')
print('\nBy vendor profile:')
print(ambiguous['profile'].value_counts())

print('\nTop offenders:')
print(ambiguous.sort_values('n_files', ascending=False).head(15).to_string())

/explore/nobackup/people/pmontesa/.nccstmp/ipykernel_2261459/3870773266.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  n_lats   =('geometry', lambda g: g.centroid.y.round(0).nunique()),
/explore/nobackup/people/pmontesa/.nccstmp/ipykernel_2261459/3870773266.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  n_lats   =('geometry', lambda g: g.centroid.y.round(0).nunique()),
/explore/nobackup/people/pmontesa/.nccstmp/ipykernel_2261459/3870773266.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  n_lats   =('geometry', lambda g: g.centroid.y.round(0).nunique()),
/exp

Ambiguous: 76

By vendor profile:
airbus_dimap    64
legion          12
Name: profile, dtype: int64

Top offenders:
                                         acquisition_id  n_sensors  n_dates  n_lats sensors  n_files       profile
8                                  200009817627_01_P001          1        1       2    LG01       82        legion
5                                  200009817622_01_P001          1        1       2    LG03       52        legion
56                                 200010965767_01_P001          1        1       2    LG05       29        legion
55                                 200010965228_01_P001          1        1       2    LG05       26        legion
375                    ORT_SPOT6_20260226_084649700_000          1        1       2   SPOT6       24  airbus_dimap
51                                 200010964781_01_P001          1        1       2    LG06       21        legion
165  DS_PHR1A_202508241845350_FR1_PX_W116N38_0412_01731          1        1    

/explore/nobackup/people/pmontesa/.nccstmp/ipykernel_2261459/3870773266.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  n_lats   =('geometry', lambda g: g.centroid.y.round(0).nunique()),
/explore/nobackup/people/pmontesa/.nccstmp/ipykernel_2261459/3870773266.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  n_lats   =('geometry', lambda g: g.centroid.y.round(0).nunique()),
/explore/nobackup/people/pmontesa/.nccstmp/ipykernel_2261459/3870773266.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  n_lats   =('geometry', lambda g: g.centroid.y.round(0).nunique()),
/exp

In [44]:
# Pick one to investigate
sample_id = '200009817627_01_P001'   # the 82-file Legion case
sub = footprint_gdf[footprint_gdf['acquisition_id'] == sample_id]

# Centroid spread
centroids = sub.geometry.centroid
print(f'Acquisition: {sample_id}')
print(f'  Files:      {len(sub)}')
print(f'  Lat range:  {centroids.y.min():.3f} to {centroids.y.max():.3f}  '
      f'(span: {centroids.y.max() - centroids.y.min():.3f} deg)')
print(f'  Lon range:  {centroids.x.min():.3f} to {centroids.x.max():.3f}  '
      f'(span: {centroids.x.max() - centroids.x.min():.3f} deg)')

# Look at the union footprint extent
union = sub.geometry.unary_union
b = union.bounds   # (minx, miny, maxx, maxy)
ns_km = (b[3] - b[1]) * 111
ew_km = (b[2] - b[0]) * 111 * 0.785  # rough at mid-latitudes
print(f'  Union bbox: ~{ns_km:.0f} km N-S, ~{ew_km:.0f} km E-W')

# Sample paths to see if there's anything odd in the filenames
print(f'\n  Sample paths:')
for p in sub['file_path'].head(5):
    print(f'    {p.split("/")[-1]}')

Acquisition: 200009817627_01_P001
  Files:      82
  Lat range:  43.231 to 43.994  (span: 0.764 deg)
  Lon range:  5.002 to 5.122  (span: 0.120 deg)
  Union bbox: ~90 km N-S, ~14 km E-W

  Sample paths:
    25AUG25104012-P3DS_R07C1-200009817627_01_P001.TIF
    25AUG25104012-P3DS_R08C1-200009817627_01_P001.TIF
    25AUG25104012-P3DS_R13C3-200009817627_01_P001.TIF
    25AUG25104012-P3DS_R11C1-200009817627_01_P001.TIF
    25AUG25104012-P3DS_R04C2-200009817627_01_P001.TIF


/explore/nobackup/people/pmontesa/.nccstmp/ipykernel_2261459/1329767563.py:6: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = sub.geometry.centroid


In [45]:
# Compute actual N-S extent per acquisition
def ns_km(group):
    if len(group) == 0:
        return 0
    b = group.geometry.unary_union.bounds
    return (b[3] - b[1]) * 111

dup_check_v2 = (
    footprint_gdf
    .groupby('acquisition_id')
    .agg(
        n_sensors=('sensor',   'nunique'),
        n_dates  =('day',      'nunique'),
        n_files  =('file_path','count'),
        profile  =('vendor_profile','first'),
    )
)
dup_check_v2['ns_km'] = footprint_gdf.groupby('acquisition_id').apply(ns_km)

# Real concerns
truly_problematic = dup_check_v2[
    (dup_check_v2['n_sensors'] > 1) |
    (dup_check_v2['n_dates']   > 1) |
    (dup_check_v2['ns_km']     > 300)    # acquisitions spanning > 300 km N-S
]

print(f'Truly problematic acquisitions: {len(truly_problematic)}')
print(truly_problematic.sort_values('ns_km', ascending=False).head(15).to_string())

Truly problematic acquisitions: 2
                                  n_sensors  n_dates  n_files       profile        ns_km
acquisition_id                                                                          
SEN_SPOT6_20260123_045336100_000          1        1       16  airbus_dimap  2055.792718
SEN_SPOT6_20260123_045437100_000          1        1       16  airbus_dimap  2055.792718


In [46]:
pixxel_dups = footprint_gdf[footprint_gdf['vendor_profile'] == 'pixxel']
pix_check = pixxel_dups.groupby('acquisition_id').agg(
    n_sensors=('sensor', 'nunique'),
    sensors  =('sensor', lambda x: ','.join(sorted(set(x))))
)
shared = pix_check[pix_check['n_sensors'] > 1]
print(f'Pixxel acquisition_ids shared across sensors: {len(shared)}')
print(shared.head())

Pixxel acquisition_ids shared across sensors: 0
Empty DataFrame
Columns: [n_sensors, sensors]
Index: []


In [47]:
sub = footprint_gdf[footprint_gdf['acquisition_id'] == '0000002215']
print(sub[['file_path', 'sensor', 'sensor_id_raw', 'vendor_profile', 'has_metadata']])

Empty DataFrame
Columns: [file_path, sensor, sensor_id_raw, vendor_profile, has_metadata]
Index: []


In [48]:
sub = footprint_gdf[footprint_gdf['acquisition_id'] == 'SEN_SPOT6_20260123_045336100_000']
print(f'Files: {len(sub)}')

# Get unique filenames and paths
print('\nUnique base paths and filenames:')
for _, row in sub.iterrows():
    fn = row['file_path'].split('/')[-1]
    parent = '/'.join(row['file_path'].split('/')[-3:-1])
    print(f'  .../{parent}/{fn}')

Files: 16

Unique base paths and filenames:
  .../VOL_SPOT6_001_A/IMG_SPOT6_MS_001_A/IMG_SPOT6_MS_202601230453359_SEN_7674265101_R1C1.TIF
  .../VOL_SPOT6_001_A/IMG_SPOT6_MS_001_A/IMG_SPOT6_MS_202601230453359_SEN_7674265101_R1C2.TIF
  .../VOL_SPOT6_001_A/IMG_SPOT6_MS_001_A/IMG_SPOT6_MS_202601230453359_SEN_7674265101_R2C1.TIF
  .../VOL_SPOT6_001_A/IMG_SPOT6_MS_001_A/IMG_SPOT6_MS_202601230453359_SEN_7674265101_R2C2.TIF
  .../VOL_SPOT6_001_A/IMG_SPOT6_P_001_A/IMG_SPOT6_P_202601230453359_SEN_7674265101_R1C1.TIF
  .../VOL_SPOT6_001_A/IMG_SPOT6_P_001_A/IMG_SPOT6_P_202601230453359_SEN_7674265101_R1C2.TIF
  .../VOL_SPOT6_001_A/IMG_SPOT6_P_001_A/IMG_SPOT6_P_202601230453359_SEN_7674265101_R2C1.TIF
  .../VOL_SPOT6_001_A/IMG_SPOT6_P_001_A/IMG_SPOT6_P_202601230453359_SEN_7674265101_R2C2.TIF
  .../VOL_SPOT6_001_A/IMG_SPOT6_MS_001_A/IMG_SPOT6_MS_202601230453359_SEN_7674265101_R1C1.TIF
  .../VOL_SPOT6_001_A/IMG_SPOT6_MS_001_A/IMG_SPOT6_MS_202601230453359_SEN_7674265101_R1C2.TIF
  .../VOL_SPOT6_001_A/IM

In [49]:
sub = footprint_gdf[footprint_gdf['acquisition_id'] == 'SEN_SPOT6_20260123_045336100_000']
print('Full paths:')
for p in sub['file_path']:
    print(f'  {p}')

Full paths:
  /explore/nobackup/projects/CSDA_eval/Airbus/SPOT/shadnagar/PROD_SPOT6_001/VOL_SPOT6_001_A/IMG_SPOT6_MS_001_A/IMG_SPOT6_MS_202601230453359_SEN_7674265101_R1C1.TIF
  /explore/nobackup/projects/CSDA_eval/Airbus/SPOT/shadnagar/PROD_SPOT6_001/VOL_SPOT6_001_A/IMG_SPOT6_MS_001_A/IMG_SPOT6_MS_202601230453359_SEN_7674265101_R1C2.TIF
  /explore/nobackup/projects/CSDA_eval/Airbus/SPOT/shadnagar/PROD_SPOT6_001/VOL_SPOT6_001_A/IMG_SPOT6_MS_001_A/IMG_SPOT6_MS_202601230453359_SEN_7674265101_R2C1.TIF
  /explore/nobackup/projects/CSDA_eval/Airbus/SPOT/shadnagar/PROD_SPOT6_001/VOL_SPOT6_001_A/IMG_SPOT6_MS_001_A/IMG_SPOT6_MS_202601230453359_SEN_7674265101_R2C2.TIF
  /explore/nobackup/projects/CSDA_eval/Airbus/SPOT/shadnagar/PROD_SPOT6_001/VOL_SPOT6_001_A/IMG_SPOT6_P_001_A/IMG_SPOT6_P_202601230453359_SEN_7674265101_R1C1.TIF
  /explore/nobackup/projects/CSDA_eval/Airbus/SPOT/shadnagar/PROD_SPOT6_001/VOL_SPOT6_001_A/IMG_SPOT6_P_001_A/IMG_SPOT6_P_202601230453359_SEN_7674265101_R1C2.TIF
  /explo

In [50]:
sub = footprint_gdf[footprint_gdf['acquisition_id'] == 'SEN_SPOT6_20260123_045336100_000']

import pandas as pd
print('Centroids of all 16 files:')
for _, row in sub.iterrows():
    c = row.geometry.centroid
    p = row['file_path']
    print(f'  ({c.y:7.3f}, {c.x:7.3f})  {p.split("/")[-1]}')

Centroids of all 16 files:
  ( 17.046,  78.183)  IMG_SPOT6_MS_202601230453359_SEN_7674265101_R1C1.TIF
  ( 17.046,  78.183)  IMG_SPOT6_MS_202601230453359_SEN_7674265101_R1C2.TIF
  ( 17.046,  78.183)  IMG_SPOT6_MS_202601230453359_SEN_7674265101_R2C1.TIF
  ( 17.046,  78.183)  IMG_SPOT6_MS_202601230453359_SEN_7674265101_R2C2.TIF
  ( 17.046,  78.183)  IMG_SPOT6_P_202601230453359_SEN_7674265101_R1C1.TIF
  ( 17.046,  78.183)  IMG_SPOT6_P_202601230453359_SEN_7674265101_R1C2.TIF
  ( 17.046,  78.183)  IMG_SPOT6_P_202601230453359_SEN_7674265101_R2C1.TIF
  ( 17.046,  78.183)  IMG_SPOT6_P_202601230453359_SEN_7674265101_R2C2.TIF
  ( 35.069, -106.613)  IMG_SPOT6_MS_202601230453359_SEN_7674265101_R1C1.TIF
  ( 35.069, -106.613)  IMG_SPOT6_MS_202601230453359_SEN_7674265101_R1C2.TIF
  ( 35.069, -106.613)  IMG_SPOT6_MS_202601230453359_SEN_7674265101_R2C1.TIF
  ( 35.069, -106.613)  IMG_SPOT6_MS_202601230453359_SEN_7674265101_R2C2.TIF
  ( 35.069, -106.613)  IMG_SPOT6_P_202601230453359_SEN_7674265101_R1C1.TI

In [51]:
sub = footprint_gdf[footprint_gdf['acquisition_id'] == 'SEN_SPOT6_20260123_045336100_000']

# Print the metadata path for each — are they all the same XML, or different XMLs?
print('Metadata paths (unique):')
for p in sorted(sub['metadata_path'].unique()):
    print(f'  {p}')

Metadata paths (unique):
  /explore/nobackup/projects/CSDA_eval/Airbus/PROD_SPOT6_001/VOL_SPOT6_001_A/IMG_SPOT6_MS_001_A/DIM_SPOT6_MS_202601230453359_SEN_7674265101.XML
  /explore/nobackup/projects/CSDA_eval/Airbus/PROD_SPOT6_001/VOL_SPOT6_001_A/IMG_SPOT6_P_001_A/DIM_SPOT6_P_202601230453359_SEN_7674265101.XML
  /explore/nobackup/projects/CSDA_eval/Airbus/SPOT/shadnagar/PROD_SPOT6_001/VOL_SPOT6_001_A/IMG_SPOT6_MS_001_A/DIM_SPOT6_MS_202601230453359_SEN_7674265101.XML
  /explore/nobackup/projects/CSDA_eval/Airbus/SPOT/shadnagar/PROD_SPOT6_001/VOL_SPOT6_001_A/IMG_SPOT6_P_001_A/DIM_SPOT6_P_202601230453359_SEN_7674265101.XML


In [52]:
# # Define sites to plot
# sites_to_plot = sorted([site for site in footprint_gdf_acquisitions_sites.Site_Primary.unique() if site == 'Boston'])
# len(sites_to_plot)

In [53]:
# Create grid
n_sites = len(sites_to_plot)
n_cols = 7
n_rows = int(np.ceil(n_sites / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols,5 * n_rows))

# Flatten axes
if n_sites == 1:
    axes = [axes]
else:
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes

# Plot each site
for idx, site_name in enumerate(sites_to_plot):
    csdalib.plot_site_coverage(site_name, 
                      footprint_gdf_acquisitions_sites, 
                      sites_gdf, 
                      #0, 
                       BUF_KM_TOTAL_FOR_DISPLAY,
                      sites_buf_gdf=sites_gdf_buf_display,
                      ax=axes[idx])  # Pass the axis here

# Hide unused subplots
for idx in range(n_sites, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/geopandas/geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/geopandas/geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/geopandas/geodataframe.py:1443: Setti

Error in callback <function _draw_all_if_interactive at 0x153fd0ea9c10> (for post_execute):
Error in callback <function flush_figures at 0x153fb880b700> (for post_execute):



KeyboardInterrupt


KeyboardInterrupt



## Create Comprehensive Summary of Multispectral Acquisitions

#### merging Acquisitions with Requested Acquisitions

how='left' (Recommended for most cases)  

+ Keeps all rows from summary_by_site (actual data you have)
+ Adds requested_acqs information where it matches
+ Use this if: You want to see all the data you actually collected, with requested acquisition counts added where available  
Result: Shows what you have, annotated with what was requested  

how='outer' (What you're currently using)  

+ Keeps all rows from both dataframes
+ Use this if: You want to see both what you collected AND what was requested (even if you didn't collect it)  
Result: Can show gaps - sites/sensors you were supposed to collect but didn't  

how='right'  

+ Keeps all rows from requested_acqs
+ Use this if: You only care about the requested combinations and want to see which ones you fulfilled  
Result: Only shows requested combinations, with actual data where available  

how='inner'  

+ Only keeps rows that match in both dataframes
+ Use this if: You only want to see requested combinations that you actually collected  
Result: Excludes both unrequested data you collected AND requested data you didn't collect  

In [ ]:
requested_acqs[requested_acqs.Site == 'Baotou']

In [ ]:
importlib.reload(csdalib)

In [ ]:
# Filter out panchromatic if needed
footprint_filtered_multispec = footprint_gdf_acquisitions_sites[footprint_gdf_acquisitions_sites.image_type != 'P']

# Step 2: Create comprehensive summaries
summaries = csdalib.create_comprehensive_summary(
    footprint_filtered_multispec,
    acq_site_mapping,
    site_name_col='Site_Primary',
    exclude_sites='Not CSDA Eval Site'
)

# Access different summaries:
site_summary            = summaries['by_site']  # Primary site assignments (no double counting)
sensor_summary          = summaries['by_sensor']  # Total by sensor
affiliation_summary     = summaries['by_affiliation']  # Total by affiliation
constellation_summary   = summaries['by_constellation']  # Total by constellation
multi_site              = summaries['multi_site_acquisitions']  # Acquisitions covering multiple sites
all_site_associations   = summaries['by_site_all_associations']  # All site-acquisition relationships

# Step 3: Merge with requested acquisitions
final_summary = pd.merge(
    #site_summary,
    site_summary.groupby(['Site_Primary','affiliation','constellation','sensor']).agg(sum).reset_index(),
    requested_acqs,
    left_on=['Site_Primary', 'affiliation', 'constellation', 'sensor'],
    right_on=['Site', 'affiliation', 'constellation', 'sensor'],
    how='outer'
)#.drop(columns=[
##    #'Site', 
#    'Affiliation', 'Constellation/Platform', 'Sensor/Generation'])

final_summary['acquisition_count'] = final_summary['acquisition_count'].fillna(0).astype(int)

final_summary['Acquisitions Requested'] = final_summary['Acquisitions Requested'].fillna(0).astype(int)
final_summary['Acquisitions Remaining'] = final_summary['Acquisitions Requested'] - final_summary['acquisition_count']
final_summary.loc[final_summary['Acquisitions Remaining'] < 0, 'Acquisitions Remaining'] = 0
out_csv_fn = f'/explore/nobackup/projects/CSDA_eval/summaries/CSDA_eval_sites_footprint_summary_MS_{today_date_string}.csv'
print(f"Saving: {out_csv_fn}")
final_summary.to_csv(out_csv_fn)

## Create `latest` footprint `gpkg` and summary `csv` files

In [ ]:
importlib.reload(csdalib)

In [ ]:
import os
import glob

# Usage example:
dir_list = ['/explore/nobackup/projects/CSDA_eval/summaries',
            '/explore/nobackup/projects/CSDA_eval/footprints'
           ]

for directory in dir_list:
    # Delete existing *_latest.csv and *_latest.gpkg files
    for ext in ['csv', 'gpkg']:
        latest_pattern = os.path.join(directory, f'*_latest.{ext}')
        existing_latest = glob.glob(latest_pattern)
        for file in existing_latest:
            try:
                os.remove(file)
                print(f"Deleted existing file: {os.path.basename(file)}")
            except Exception as e:
                print(f"Warning: Could not delete {file}: {e}")
    
    # Now copy the latest files
    latest_files = csdalib.copy_all_latest_files(directory, extensions=['csv', 'gpkg'])

    cmd = f'chmod -R 755 {directory}'
    print(f'\n\tOpening up access with chmod: {cmd}')
    !eval $cmd
    
    # Show results
    print("\n=== Summary ===")
    for key, filepath in latest_files.items():
        print(f"{key}: {os.path.basename(filepath)}")